# Can Neural GPUs Learn Arithmetic Algorithms & Generalize Beyond Training Length?

In [1]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

## 1. Create a Binary-Addition Generator

In [2]:
FIXED_LENGTH = 8

class ValidateBinary(Exception):
  pass

class PaddingException(Exception):
  pass

def validate_binary(num: str):
  if num and set(num).issubset({'0', '1'}):
    return True
  raise ValidateBinary(
      "Binary numbers are only 0 and 1."
  )

def padding(num: str, length: int):
  if len(num) > length:
    raise PaddingException(
        f"Number must be at most {length} bits."
    )
  return num.zfill(length)

def binary_addition(a: str, b: str):
  validate_binary(a)
  validate_binary(b)

  a = padding(a, FIXED_LENGTH)
  b = padding(b, FIXED_LENGTH)

  result = bin(int(a, 2) + int(b, 2))[2:]

  return padding(result, FIXED_LENGTH + 1)

a = "00011111"
b = "1010101"
binary_addition(a, b)

'001110100'

### 1.1 Generator Many Binary Addition Samples

In [3]:
import random

def generate_sample(bit_length):
    max_value = 2**bit_length - 1

    a = random.randint(0, max_value)
    b = random.randint(0, max_value)

    a_binary = bin(a)[2:].zfill(bit_length)
    b_binary = bin(b)[2:].zfill(bit_length)

    result = bin(a + b)[2:].zfill(bit_length + 1)

    return a_binary, b_binary, result

a, b, result = generate_sample(FIXED_LENGTH)
print(a, b, result)

00111001 00001100 001000101


In [4]:
def generate_dataset(n_samples, bit_length):
    return [
        generate_sample(bit_length)
        for _ in range(n_samples)
    ]

dataset = generate_dataset(1000, FIXED_LENGTH)

dataset[:5]

[('10001100', '01111101', '100001001'),
 ('01110010', '01000111', '010111001'),
 ('00110100', '00101100', '001100000'),
 ('11011000', '00010000', '011101000'),
 ('00001111', '00101111', '000111110')]

### 1.2 Convert Each Binary String into Tensor

In [5]:
import torch

def binary_to_tensor(binary: str):
    return torch.tensor(
        [int(bit) for bit in binary[::-1]],
        dtype=torch.long
    )

a, b, result = generate_sample(FIXED_LENGTH)

a_tensor = binary_to_tensor(a)
b_tensor = binary_to_tensor(b)
result_tensor = binary_to_tensor(result)

print(a_tensor)
print(b_tensor)
print(result_tensor)

print(a_tensor.shape)
print(result_tensor.shape)

tensor([1, 1, 0, 1, 1, 1, 1, 0])
tensor([1, 1, 0, 0, 1, 1, 1, 0])
tensor([0, 1, 1, 1, 0, 1, 1, 1, 0])
torch.Size([8])
torch.Size([9])


In [6]:
x = torch.stack([a_tensor, b_tensor], dim=1)
print(x.shape)

print(x[0])

torch.Size([8, 2])
tensor([1, 1])


## 2. Create PyTorch Dataset

In [7]:
from torch.utils.data import Dataset

class BinaryAdditionDataset(Dataset):
    def __init__(self, n_samples, bit_length):
        self.data = generate_dataset(
            n_samples,
            bit_length
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        a, b, result = self.data[idx]

        a_tensor = binary_to_tensor(a)
        b_tensor = binary_to_tensor(b)
        y = binary_to_tensor(result)

        x = torch.stack(
            [a_tensor, b_tensor],
            dim=1
        )

        return x, y

In [8]:
dataset = BinaryAdditionDataset(
    1000,
    bit_length=FIXED_LENGTH
)
x, y = dataset[0]

print(x)
print(y)

print(f"x shape is {x.shape}")
print(f"y shape is {y.shape}")

tensor([[1, 1],
        [0, 0],
        [0, 0],
        [1, 1],
        [1, 0],
        [0, 0],
        [0, 1],
        [1, 0]])
tensor([0, 1, 0, 0, 0, 1, 1, 1, 0])
x shape is torch.Size([8, 2])
y shape is torch.Size([9])


### 2.1 Create a DataLoader

In [9]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

x_batch, y_batch = next(iter(loader))

print(x_batch.shape)
print(y_batch.shape)

torch.Size([32, 8, 2])
torch.Size([32, 9])
